# Projeto IA - Unidade 1
## Dia 3: Modelagem Não Supervisionada (PCA, K-Means e DBSCAN)

Neste notebook, executamos a redução de dimensionalidade e a busca por agrupamentos ocultos em **AMBOS** os conjuntos de dados, extraindo métricas de qualidade.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Algoritmos Não Supervisionados
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN

# Métricas de Agrupamento
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# Pre-processamento rápido (Recap do Dia 1 e 2)
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Recarregando e Padronizando os Dados

In [ ]:
# 1. Breast Cancer (Classificação)
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target # Usaremos para validar métricas externas (ARI/NMI)
scaler_cancer = StandardScaler()
X_cancer_scaled = scaler_cancer.fit_transform(X_cancer)

# 2. California Housing (Regressão)
housing = fetch_california_housing()
X_housing = housing.data
y_housing = housing.target # Alvo contínuo, usaremos apenas para colorir o gráfico PCA
scaler_housing = StandardScaler()
X_housing_scaled = scaler_housing.fit_transform(X_housing)

print("Dados carregados e padronizados com sucesso!")

---
## PARTE A: Análise no Dataset de CLASSIFICAÇÃO (Breast Cancer)

In [ ]:
# PCA: Variância Explicada Acumulada
pca_cancer = PCA().fit(X_cancer_scaled)
variancia_acumulada_c = np.cumsum(pca_cancer.explained_variance_ratio_)

plt.plot(range(1, len(variancia_acumulada_c) + 1), variancia_acumulada_c, marker='o', color='purple')
plt.title('PCA Breast Cancer - Variância Explicada Acumulada')
plt.xlabel('Componentes Principais')
plt.ylabel('Variância Retida')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% de Variância')
plt.legend()
plt.show()

# PCA: Projeção 2D
X_cancer_pca2d = PCA(n_components=2).fit_transform(X_cancer_scaled)
plt.scatter(X_cancer_pca2d[:, 0], X_cancer_pca2d[:, 1], c=y_cancer, cmap='coolwarm', alpha=0.7)
plt.title('Projeção 2D (PCA) - Breast Cancer (Maligno vs Benigno)')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.show()

In [ ]:
# Método do Cotovelo (K-Means)
inercias_c = []
Ks = range(2, 11)
for k in Ks:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_cancer_scaled)
    inercias_c.append(kmeans.inertia_)

plt.plot(Ks, inercias_c, marker='s', color='green')
plt.title('Método do Cotovelo (Elbow Method) - Breast Cancer')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.axvline(x=2, color='red', linestyle='--', label='K=2 (Classes Biológicas Reais)')
plt.legend()
plt.show()

In [ ]:
# Treinando os modelos e capturando as métricas - Breast Cancer
kmeans_c = KMeans(n_clusters=2, init='k-means++', random_state=42, n_init=10)
labels_kmeans_c = kmeans_c.fit_predict(X_cancer_scaled)

dbscan_c = DBSCAN(eps=2.5, min_samples=5)
labels_dbscan_c = dbscan_c.fit_predict(X_cancer_scaled)

# Filtro para métricas internas do DBSCAN (ignorar o ruído = -1)
mask_c = labels_dbscan_c != -1

res_cancer = {
    "Métrica": ["Inércia", "Silhouette", "Calinski-Harabasz", "Davies-Bouldin", "ARI (Ext)", "NMI (Ext)"],
    "K-Means": [
        kmeans_c.inertia_,
        silhouette_score(X_cancer_scaled, labels_kmeans_c),
        calinski_harabasz_score(X_cancer_scaled, labels_kmeans_c),
        davies_bouldin_score(X_cancer_scaled, labels_kmeans_c),
        adjusted_rand_score(y_cancer, labels_kmeans_c),
        normalized_mutual_info_score(y_cancer, labels_kmeans_c)
    ],
    "DBSCAN": [
        "N/A",
        silhouette_score(X_cancer_scaled[mask_c], labels_dbscan_c[mask_c]) if len(np.unique(labels_dbscan_c[mask_c])) > 1 else "N/A",
        calinski_harabasz_score(X_cancer_scaled[mask_c], labels_dbscan_c[mask_c]) if len(np.unique(labels_dbscan_c[mask_c])) > 1 else "N/A",
        davies_bouldin_score(X_cancer_scaled[mask_c], labels_dbscan_c[mask_c]) if len(np.unique(labels_dbscan_c[mask_c])) > 1 else "N/A",
        adjusted_rand_score(y_cancer, labels_dbscan_c),
        normalized_mutual_info_score(y_cancer, labels_dbscan_c)
    ]
}
print("Tabela Comparativa de Agrupamento - CLASSIFICAÇÃO")
display(pd.DataFrame(res_cancer))

---
## PARTE B: Análise no Dataset de REGRESSÃO (California Housing)

In [ ]:
# PCA: Variância Explicada Acumulada
pca_housing = PCA().fit(X_housing_scaled)
variancia_acumulada_h = np.cumsum(pca_housing.explained_variance_ratio_)

plt.plot(range(1, len(variancia_acumulada_h) + 1), variancia_acumulada_h, marker='o', color='orange')
plt.title('PCA California Housing - Variância Explicada Acumulada')
plt.xlabel('Componentes Principais')
plt.ylabel('Variância Retida')
plt.axhline(y=0.90, color='r', linestyle='--', label='90% de Variância')
plt.legend()
plt.show()

# PCA: Projeção 2D (As cores representam o Preço Mediano contínuo)
X_housing_pca2d = PCA(n_components=2).fit_transform(X_housing_scaled)
scatter = plt.scatter(X_housing_pca2d[:, 0], X_housing_pca2d[:, 1], c=y_housing, cmap='viridis', alpha=0.5)
plt.title('Projeção 2D (PCA) - California Housing')
plt.xlabel('Componente Principal 1')
plt.ylabel('Componente Principal 2')
plt.colorbar(scatter, label='Preço Mediano (MedHouseVal)')
plt.show()

In [ ]:
# Método do Cotovelo (K-Means) em uma amostragem (para não demorar em 20k linhas)
inercias_h = []
Ks = range(2, 11)
for k in Ks:
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(X_housing_scaled)
    inercias_h.append(kmeans.inertia_)

plt.plot(Ks, inercias_h, marker='s', color='blue')
plt.title('Método do Cotovelo (Elbow Method) - California Housing')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia')
plt.show()
# Vamos adotar K=3 para o K-Means (dividir casas em 3 perfis geográficos/estruturais)

In [ ]:
# Treinando os modelos e capturando as métricas - California Housing
# Nota: Métricas Externas (ARI, NMI) não existem aqui pois o alvo é preço (contínuo), não é classe.

kmeans_h = KMeans(n_clusters=3, init='k-means++', random_state=42, n_init=10)
labels_kmeans_h = kmeans_h.fit_predict(X_housing_scaled)

dbscan_h = DBSCAN(eps=1.0, min_samples=10)
labels_dbscan_h = dbscan_h.fit_predict(X_housing_scaled)

# Filtro para métricas internas do DBSCAN (ignorar o ruído = -1)
mask_h = labels_dbscan_h != -1

res_housing = {
    "Métrica": ["Inércia", "Silhouette", "Calinski-Harabasz", "Davies-Bouldin"],
    "K-Means": [
        kmeans_h.inertia_,
        silhouette_score(X_housing_scaled, labels_kmeans_h),
        calinski_harabasz_score(X_housing_scaled, labels_kmeans_h),
        davies_bouldin_score(X_housing_scaled, labels_kmeans_h)
    ],
    "DBSCAN": [
        "N/A",
        silhouette_score(X_housing_scaled[mask_h], labels_dbscan_h[mask_h]) if len(np.unique(labels_dbscan_h[mask_h])) > 1 else "N/A",
        calinski_harabasz_score(X_housing_scaled[mask_h], labels_dbscan_h[mask_h]) if len(np.unique(labels_dbscan_h[mask_h])) > 1 else "N/A",
        davies_bouldin_score(X_housing_scaled[mask_h], labels_dbscan_h[mask_h]) if len(np.unique(labels_dbscan_h[mask_h])) > 1 else "N/A"
    ]
}
print("Tabela Comparativa de Agrupamento - REGRESSÃO")
display(pd.DataFrame(res_housing))